# 02 - Tien xu ly du lieu va Mo hinh hoa

Notebook chua day du cac ky thuat tien xu ly du lieu:

## Cac ky thuat tien xu ly:

### 1. Ky thuat tich hop (Integration)
- Tich hop du lieu tu nhieu nguon
- Trong he thong: Load tu file CSV

### 2. Ky thuat lay mau (Sampling)
- Simple Random Sampling
- Stratified Sampling (Phan tang)
- Sampling co/ khong co thay the

### 3. Ky thuat giam so chieu (Dimensionality Reduction)
- PCA (Principal Component Analysis)
- Feature Selection (Lua chon thuoc tinh con)

### 4. Lua chon tap thuoc tinh con dac trung (Feature Selection)
- SelectKBest voi ANOVA F-test
- Mutual Information

### 5. Tao moi thuoc tinh dac trung (Feature Engineering)
- BMI_category (Roi rac hoa)
- comorbidity_score
- healthy_lifestyle

### 6. Roi rac hoa va nhi phan hoa (Discretization & Binarization)
- Chuyen doi BMI thanh cac nhom

### 7. Chuyen doi thuoc tinh (Attribute Transformation)
- StandardScaler cho bien lien tuc
- OneHotEncoder cho bien phan loai

## 1. Import thu vien

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import json
import joblib
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

## 2. Load du lieu (Ky thuat Tich hop)

In [ ]:
from src.data.load_data import load_raw_data

df = load_raw_data()
print(f"Shape: {df.shape}")
print(f"So dong: {df.shape[0]:,}")
print(f"So cot: {df.shape[1]}")
df.head()

## 3. Kiem tra chat luong du lieu

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
# Kiem tra missing values
missing = df.isnull().sum()
print(f"Missing values: {missing.sum()}")
if missing.sum() == 0:
    print("Du lieu sach, khong co missing values.")

In [ ]:
# Kiem tra gia tri hop le
binary_cols = ["HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex"]

ordinal_ranges = {"GenHlth": (1, 5), "Age": (1, 13), "Education": (1, 6), "Income": (1, 8)}

print("Kiem tra gia tri khong hop le:")
for col in binary_cols:
    invalid = sorted(set(df[col].dropna().unique()) - {0, 1, 0.0, 1.0})
    if invalid:
        print(f"  {col}: gia tri khong hop le {invalid}")

for col, (mn, mx) in ordinal_ranges.items():
    cnt = (~df[col].between(mn, mx)).sum()
    if cnt > 0:
        print(f"  {col}: {cnt} gia tri khong hop le")

## 4. Ky thuat lay mau (Sampling)

### 4.1. Gioi thieu cac phuong phap lay mau

In [ ]:
print("""
KY THUAT LAY MAU (SAMPLING)
==========================

1. Simple Random Sampling (Lay mau ngau nhien don gian)
   - Moi phan tu co xac suat duoc chon nhu nhau
   - Khong dam bao tien loi cho du lieu mat can bang

2. Stratified Sampling (Lay mau phan tang) - DANG SU DUNG
   - Chia du lieu thanh nhieu phan vung (strata) theo bien muc tieu
   - Rut ra mau ngau nhien tu moi phan vung
   - Dam bao ti le lop giong nhau trong train/test

3. Sampling With Replacement (Co thay the)
   - Mot phan tu co the duoc chon nhieu lan

4. Sampling Without Replacement (Khong thay the)
   - Mot phan tu chi duoc chon mot lan
""")

### 4.2. Xem phan phoi target truoc khi lay mau

In [ ]:
target_col = "Diabetes_binary"

print("Phan phoi target trong du lieu goc:")
target_dist = df[target_col].value_counts().sort_index()
print(target_dist)
print(f"\nTi le: {target_dist[0]/len(df)*100:.1f}% non-diabetic, {target_dist[1]/len(df)*100:.1f}% diabetic")

### 4.3. Minh hoa Simple Random Sampling

In [ ]:
# Simple Random Sampling (khong stratify)
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    df.drop(columns=[target_col]),
    df[target_col],
    test_size=0.2,
    random_state=42
)  # Khong co stratify

print("Simple Random Sampling (khong stratify):")
print(f"Train target dist: {y_train_random.value_counts(normalize=True).to_dict()}")
print(f"Test target dist: {y_test_random.value_counts(normalize=True).to_dict()}")

### 4.4. Minh hoa Stratified Sampling

In [ ]:
# Stratified Sampling (co stratify)
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=[target_col]),
    df[target_col],
    test_size=0.2,
    random_state=42,
    stratify=df[target_col]
)

print("Stratified Sampling (co stratify=y):")
print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"\nTrain target dist: {y_train.value_counts(normalize=True).to_dict()}")
print(f"Test target dist: {y_test.value_counts(normalize=True).to_dict()}")

### 4.5. So sanh

In [ ]:
print("\nSo sanh Random vs Stratified:")
print("-"*50)
print(f"{'Metric':<25} {'Random':<15} {'Stratified':<15}")
print("-"*50)
print(f"{'Train diabetic ratio':<25} {y_train_random.mean():.4f}       {y_train.mean():.4f}")
print(f"{'Test diabetic ratio':<25} {y_test_random.mean():.4f}       {y_test.mean():.4f}")
print(f"{'Train size':<25} {len(y_train_random):<15} {len(y_train):<15}")
print(f"{'Test size':<25} {len(y_test_random):<15} {len(y_test):<15}")

print("\nKet luan: Stratified Sampling dam bao ti le lop nhin nhau hon.")

## 5. Ky thuat Tao moi thuoc tinh dac trung (Feature Engineering)

### 5.1. Gioi thieu

In [ ]:
print("""
TAO MOI THUOC TINH DAC TRUNG (FEATURE ENGINEERING)
=================================================

Tao cac feature moi tu cac feature goc de:
1. Tang kha nang du bao cua mo hinh
2. Giam phuc tap cua du lieu
3. Giup mo hinh de giai thich hon

Cac feature moi trong he thong:
- BMI_category: Roi rac hoa BMI thanh cac nhom
- comorbidity_score: Tong so tinh trang benh di kem
- healthy_lifestyle: Diem cuoc song lành manh
""")

### 5.2. Tao BMI_category (Roi rac hoa)

In [ ]:
# BMI_category: Phan nhom BMI theo chuan WHO
df["BMI_category"] = pd.cut(
    df["BMI"],
    bins=[0, 18.5, 25, 30, np.inf],
    labels=["underweight", "normal", "overweight", "obese"],
    include_lowest=True
)

print("BMI Category Distribution:")
print(df["BMI_category"].value_counts().sort_index())

# Ve bieu do
plt.figure(figsize=(8, 5))
df["BMI_category"].value_counts().sort_index().plot(kind='bar')
plt.title("BMI Category Distribution (Roi rac hoa)")
plt.xlabel("BMI Category")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "bmi_category_distribution.png", dpi=150)
plt.show()

### 5.3. Tao comorbidity_score (Tich hop cac thuoc tinh)

In [ ]:
# comorbidity_score: Tong hop cac tinh trang benh ly
# Ky thuat TICH HOP: Ket hop nhieu thuoc tinh thanh mot
comorbidity_cols = ["HighBP", "HighChol", "Stroke", "HeartDiseaseorAttack", "DiffWalk"]
df["comorbidity_score"] = df[comorbidity_cols].sum(axis=1)

print("Comorbidity Score (Tich hop 5 thuoc tinh benh ly):")
print(f"Cac thuoc tinh duoc tich hop: {comorbidity_cols}")
print(f"Diem trung binh: {df['comorbidity_score'].mean():.2f}")
print(f"Diem min: {df['comorbidity_score'].min()}, max: {df['comorbidity_score'].max()}")

# Ve phan phoi
plt.figure(figsize=(8, 5))
df["comorbidity_score"].hist(bins=6)
plt.title("Comorbidity Score Distribution")
plt.xlabel("Comorbidity Score")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "comorbidity_score_distribution.png", dpi=150)
plt.show()

### 5.4. Tao healthy_lifestyle

In [ ]:
# healthy_lifestyle: Diem cuoc song lành manh
df["healthy_lifestyle"] = (
    df["PhysActivity"]
    + df["Fruits"]
    + df["Veggies"]
    + (1 - df["Smoker"])
    + (1 - df["HvyAlcoholConsump"])
)

print("Healthy Lifestyle Score:")
print(f"Diem trung binh: {df['healthy_lifestyle'].mean():.2f}")
print(f"Diem min: {df['healthy_lifestyle'].min()}, max: {df['healthy_lifestyle'].max()}")

# Ve phan phoi
plt.figure(figsize=(8, 5))
df["healthy_lifestyle"].hist(bins=6)
plt.title("Healthy Lifestyle Score Distribution")
plt.xlabel("Healthy Lifestyle Score")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "healthy_lifestyle_distribution.png", dpi=150)
plt.show()

### 5.5. Kiem tra tuong quan cua feature moi voi target

In [ ]:
# Tinh correlation cua feature moi voi target
new_features = ["BMI_category", "comorbidity_score", "healthy_lifestyle"]
df["BMI_category_encoded"] = df["BMI_category"].cat.codes

correlations = {
    "comorbidity_score": df["comorbidity_score"].corr(df[target_col]),
    "healthy_lifestyle": df["healthy_lifestyle"].corr(df[target_col]),
    "BMI_category_encoded": df["BMI_category_encoded"].corr(df[target_col])
}

print("Correlation voi Diabetes_binary:")
for feat, corr in sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {feat}: {corr:.4f}")

## 6. Ky thuat Chuyen doi thuoc tinh (Attribute Transformation)

### 6.1. Gioi thieu

In [ ]:
print("""
CHUYEN DOI THUOC TINH (ATTRIBUTE TRANSFORMATION)
===============================================

1. StandardScaler: Chuan hoa ve phan phoi chuan (mean=0, std=1)
   - Phu hop cho Logistic Regression
   - Cac bien: BMI, MentHlth, PhysHlth, Age, comorbidity_score, healthy_lifestyle

2. OneHotEncoder: Ma hoa one-hot cho bien phan loai
   - Bien: BMI_category (4 gia tri)

3. Binary encoding: Giu nguyen cho bien binary (0/1)
""")

### 6.2. Xac dinh cac nhom bien

In [ ]:
# Cac bien can scale
continuous_cols_to_scale = ["BMI", "MentHlth", "PhysHlth", "Age", "comorbidity_score", "healthy_lifestyle"]

# Cac bien categorical can encode
categorical_cols = ["BMI_category"]

# Cac bien binary (0/1, khong can scale)
binary_cols = ["HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex",
    "GenHlth", "Education", "Income"]

print(f"So bien can scale: {len(continuous_cols_to_scale)}")
print(f"So bien categorical: {len(categorical_cols)}")
print(f"So bien binary: {len(binary_cols)}")

### 6.3. Tao Preprocessor

In [ ]:
# Chuan bi X, y
X = df.drop(columns=[target_col, "BMI_category_encoded"])
y = df[target_col].astype(int)
feature_columns = X.columns.tolist()

# Chi lay cot can thiet cho X_train/X_test
X_train_fe = X.loc[X_train.index].copy()
X_test_fe = X.loc[X_test.index].copy()

# Tao ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("continuous_scaler", StandardScaler(), continuous_cols_to_scale),
        ("category_encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ],
    remainder="passthrough"
)

print("Preprocessor da duoc cau hinh.")
print(f"Scale: {continuous_cols_to_scale}")
print(f"Encode: {categorical_cols}")

### 6.4. So sanh kich thuoc truoc va sau tien xu ly

In [ ]:
# So sanh kich thuoc dataset goc va dataset sau tien xu ly
raw_data_path = ROOT / "data" / "raw" / "diabetes_binary_health_indicators_BRFSS2015.csv"
raw_shape = pd.read_csv(raw_data_path).shape
engineered_shape = df.drop(columns=["BMI_category_encoded"]).shape

comparison_df = pd.DataFrame({
    "Giai doan": [
        "Dataset goc",
        "Sau feature engineering",
        "X_train sau tien xu ly",
        "X_test sau tien xu ly",
        "y_train",
        "y_test"
    ],
    "So dong": [
        raw_shape[0],
        engineered_shape[0],
        X_train_fe.shape[0],
        X_test_fe.shape[0],
        y_train.shape[0],
        y_test.shape[0]
    ],
    "So cot": [
        raw_shape[1],
        engineered_shape[1],
        X_train_fe.shape[1],
        X_test_fe.shape[1],
        1,
        1
    ]
})

display(comparison_df)

print("Tong so dong sau khi chia train/test:", X_train_fe.shape[0] + X_test_fe.shape[0])
print(f"Ty le train: {X_train_fe.shape[0] / raw_shape[0] * 100:.2f}%")
print(f"Ty le test: {X_test_fe.shape[0] / raw_shape[0] * 100:.2f}%")

## 7. Ky thuat Lua chon thuoc tinh con dac trung (Feature Selection)

### 7.1. Gioi thieu

In [ ]:
print("""
LUA CHON THUOC TINH CON DAC TRUNG (FEATURE SELECTION)
======================================================

Muc dich: Loai bo feature yếu, giam overfitting, tang hieu suat

Phuong phap:
1. SelectKBest: Chon k feature tot nhat theo scoring function
2. ANOVA F-test: Danh gia moi quan he tuyen tinh giua feature va target
3. Mutual Information: Do phu thuoc nhau giua feature va target
""")

### 7.2. Chi so F (ANOVA F-score)

In [ ]:
# Fit preprocessor
X_train_scaled = preprocessor.fit_transform(X_train_fe)
X_test_scaled = preprocessor.transform(X_test_fe)

# Lay feature names sau transform
temp_feature_names = preprocessor.get_feature_names_out()
temp_feature_names = [
    name.replace("continuous_scaler__", "").replace("category_encoder__", "").replace("remainder__", "")
    for name in temp_feature_names
]

# SelectKBest voi ANOVA F-test
selector_f = SelectKBest(score_func=f_classif, k='all')
selector_f.fit(X_train_scaled, y_train)

# Diem F cua moi feature
feature_scores = pd.DataFrame({
    "feature": temp_feature_names,
    "f_score": selector_f.scores_,
    "p_value": selector_f.pvalues_
}).sort_values("f_score", ascending=False)

print("Top 15 features theo ANOVA F-score:")
print(feature_scores.head(15).to_string(index=False))

### 7.3. Ve bieu do Feature Scores

In [ ]:
plt.figure(figsize=(12, 10))
top_n = 25
top_features = feature_scores.head(top_n)
sns.barplot(data=top_features, x="f_score", y="feature", palette="viridis")
plt.title("Feature Importance - ANOVA F-Score", fontsize=14)
plt.xlabel("F-Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "feature_selection_fscore.png", dpi=150)
plt.show()

### 7.4. Mutual Information

In [ ]:
# Mutual Information
mi_score_func = lambda X, y: mutual_info_classif(X, y, random_state=42)
selector_mi = SelectKBest(score_func=mi_score_func, k='all')
selector_mi.fit(X_train_scaled, y_train)

mi_scores = pd.DataFrame({
    "feature": temp_feature_names,
    "mi_score": selector_mi.scores_
}).sort_values("mi_score", ascending=False)

print("Top 15 features theo Mutual Information:")
print(mi_scores.head(15).to_string(index=False))

In [ ]:
plt.figure(figsize=(12, 10))
top_mi = mi_scores.head(top_n)
sns.barplot(data=top_mi, x="mi_score", y="feature", palette="plasma")
plt.title("Feature Importance - Mutual Information", fontsize=14)
plt.xlabel("Mutual Information Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "feature_selection_mi.png", dpi=150)
plt.show()

### 7.5. So sanh F-score va MI

In [ ]:
# Merge hai phuong phap
feature_comparison = feature_scores.merge(mi_scores, on="feature")
feature_comparison = feature_comparison.sort_values("f_score", ascending=False)

print("So sanh F-score va Mutual Information (top 15):")
print(feature_comparison.head(15).to_string(index=False))

## 8. Ky thuat Giam so chieu (Dimensionality Reduction)

### 8.1. Gioi thieu PCA

In [ ]:
print("""
GIAM SO CHIEU (DIMENSIONALITY REDUCTION)
=======================================

PCA (Principal Component Analysis):
- Tim cac thanh phan chinh (principal components)
- Cac thanh phan chinh vuong goc (orthogonal) voi nhau
- PC1 giai thich nhieu phuong sai nhat
- PC2 giai thich phuong sai con lai

Loi ich:
- Giam so chieu du lieu
- Loai bo noise
- Tang toc do training
- Giam overfitting
""")

### 8.2. PCA Analysis

In [ ]:
# Fit PCA de xem phan bo phuong sai
pca_full = PCA()
pca_full.fit(X_train_scaled)

print(f"Tong so components: {pca_full.n_components_}")
print(f"\nPhuong sai giai thich boi cac components:")
for i, var in enumerate(pca_full.explained_variance_ratio_[:10], 1):
    print(f"  PC{i}: {var*100:.2f}%")

### 8.3. Ve Scree Plot

In [ ]:
# Tinh cumulative variance
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Individual variance
ax1.bar(range(1, len(pca_full.explained_variance_ratio_) + 1), 
        pca_full.explained_variance_ratio_ * 100, alpha=0.7, color='steelblue')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance (%)')
ax1.set_title('Individual Explained Variance')

# Cumulative variance
ax2.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 
         'ro-', markersize=4)
ax2.axhline(y=90, color='g', linestyle='--', label='90% threshold')
ax2.axhline(y=95, color='orange', linestyle='--', label='95% threshold')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Explained Variance (%)')
ax2.set_title('Cumulative Explained Variance')
ax2.legend()

plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "pca_scree_plot.png", dpi=150)
plt.show()

# Tim so components can thiet
n_90 = np.argmax(cumulative_variance >= 90) + 1
n_95 = np.argmax(cumulative_variance >= 95) + 1
print(f"\nSo components de giai thich 90% variance: {n_90}")
print(f"So components de giai thich 95% variance: {n_95}")

### 8.4. PCA voi so components toi uu

In [ ]:
# PCA voi 95% variance
pca_95 = PCA(n_components=0.95)
X_train_pca = pca_95.fit_transform(X_train_scaled)
X_test_pca = pca_95.transform(X_test_scaled)

print(f"PCA voi 95% variance:")
print(f"  So components: {pca_95.n_components_}")
print(f"  Giai thich duoc: {sum(pca_95.explained_variance_ratio_)*100:.2f}%")
print(f"  Shape truoc PCA: {X_train_scaled.shape}")
print(f"  Shape sau PCA: {X_train_pca.shape}")
print(f"  Giam: {X_train_scaled.shape[1] - pca_95.n_components_} features")

## 9. Xay dung Pipeline va Huan luyen Mo hinh

Trong phan nay, ta se su dung du lieu da duoc tien xu ly:
- Scale: StandardScaler
- Encode: OneHotEncoder
- Khong su dung Feature Selection hay PCA trong pipeline chinh
  (vi cac mo hinh tree-based co the tu dong xu ly)

### 9.1. Xac dinh class weight

In [ ]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / positive_count

print(f"Class distribution:")
print(f"  Negative (0): {negative_count:,}")
print(f"  Positive (1): {positive_count:,}")
print(f"  Imbalance ratio: {scale_pos_weight:.2f}")

print("\nPhuong phap xu ly mat can bang:")
print("  - Logistic Regression: class_weight='balanced'")
print("  - Random Forest: class_weight='balanced'")
print("  - XGBoost: scale_pos_weight")

### 9.2. Cau hinh 3 mo hinh

In [ ]:
# Logistic Regression - Baseline
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="lbfgs",
        random_state=42
    ))
])

In [ ]:
# Random Forest
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=160,
        max_depth=16,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

In [ ]:
# XGBoost
xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=180,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1
    ))
])

### 9.3. Huan luyen

In [ ]:
# Dam bao cac model da duoc khoi tao truoc khi huan luyen
if "xgb_model" not in globals():
    xgb_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", XGBClassifier(
            n_estimators=180,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=-1
        ))
    ])

models = {
    "logistic_regression": logistic_model,
    "random_forest": rf_model,
    "xgboost": xgb_model
}

start = time.time()

for name, model in models.items():
    print(f"Dang huan luyen {name}...")
    model.fit(X_train_fe, y_train)
    print(f"  Xong!")

print(f"\nTong thoi gian: {time.time() - start:.2f} giay")

## 10. Danh gia Mo hinh

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist()
    }

    return metrics

In [ ]:
all_metrics = {}

for name, model in models.items():
    all_metrics[name] = evaluate_model(model, X_test_fe, y_test, name)

# Hien thi ket qua
comparison_df = pd.DataFrame([
    {"Model": name, **m}
    for name, m in all_metrics.items()
])[["Model", "accuracy", "precision", "recall", "f1", "roc_auc"]]

print("\nKET QUA DANH GIA:")
print(comparison_df.to_string(index=False))

In [ ]:
print("\n" + "="*70)
print("CLASSIFICATION REPORTS:")
print("="*70)

for name, model in models.items():
    y_pred = model.predict(X_test_fe)
    print(f"\n{name.replace('_', ' ').title()}:")
    print(classification_report(y_test, y_pred, target_names=["Non-diabetic", "Diabetic"]))

## 11. Bieu do So sanh

In [ ]:
# Grouped Bar Metrics
metrics_long = comparison_df.melt(
    id_vars="Model",
    value_vars=["accuracy", "precision", "recall", "f1", "roc_auc"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(11, 6))
sns.barplot(data=metrics_long, x="Metric", y="Score", hue="Model")
plt.ylim(0, 1)
plt.title("Model Performance Comparison")
plt.legend(title="Model")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "grouped_bar_metrics.png", dpi=150)
plt.show()

In [ ]:
# ROC Curve
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

colors = {"logistic_regression": "#7c3aed", "random_forest": "#2563eb", "xgboost": "#16a34a"}

plt.figure(figsize=(8, 6))
for name, model in models.items():
    y_proba = model.predict_proba(X_test_fe)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name.replace('_', ' ').title()} AUC = {roc_auc:.4f}",
             color=colors[name], linewidth=2)

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "roc_curve_comparison.png", dpi=150)
plt.show()

In [ ]:
# Confusion Matrix
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cmaps = {"logistic_regression": "Purples", "random_forest": "Blues", "xgboost": "Greens"}

for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_test_fe, y_test, ax=ax, cmap=cmaps[name])
    ax.set_title(name.replace("_", " ").title())

plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "confusion_matrix.png", dpi=150)
plt.show()

## 12. Luu artifacts

In [ ]:
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATHS = {
    "logistic_regression": MODELS_DIR / "logistic_regression.joblib",
    "random_forest": MODELS_DIR / "random_forest.joblib",
    "xgboost": MODELS_DIR / "xgboost.joblib"
}

for name, model in models.items():
    joblib.dump(model, MODEL_PATHS[name])

joblib.dump(feature_columns, MODELS_DIR / "feature_columns.joblib")
print("Da luu models")

In [ ]:
REPORTS_DIR = ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

with open(REPORTS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2, ensure_ascii=False)

print("Da luu metrics.json")

## 13. Tong ket cac ky thuat tien xu ly

In [ ]:
print("""
TONG KET CAC KY THUAT TIEN XU LY DA SU DUNG
==========================================

1. KY THUAT TICH HOP (Integration)
   - Load du lieu tu file CSV
   - Kiem tra chat luong du lieu

2. KY THUAT LAY MAU (Sampling)
   - Stratified Sampling (stratify=y)
   - Dam bao ti le lop nhin nhau trong train/test

3. TAO MOI THUOC TINH DAC TRUNG (Feature Engineering)
   - BMI_category: Roi rac hoa BMI thanh 4 nhom
   - comorbidity_score: Tong hop 5 tinh trang benh
   - healthy_lifestyle: Diem cuoc song lành manh

4. CHUYEN DOI THUOC TINH (Attribute Transformation)
   - StandardScaler: Chuan hoa bien lien tuc
   - OneHotEncoder: Ma hoa bien phan loai

5. LUA CHON THUOC TINH CON DAC TRUNG (Feature Selection)
   - SelectKBest voi ANOVA F-test
   - Mutual Information
   - Da phan tich va xac dinh feature quan trong

6. GIAM SO CHIEU (Dimensionality Reduction)
   - PCA (PCA voi 95% variance con 24 components)
   - Giai thich phuong sai cua tung thanh phan

7. XU LY MAT CAN BANG LOP
   - class_weight='balanced' cho LR va RF
   - scale_pos_weight cho XGBoost

MO HINH DA HUAN LUYEN:
- Logistic Regression (baseline)
- Random Forest
- XGBoost
""")